<a href="https://colab.research.google.com/github/doyun1119/-/blob/main/rogue_colab_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROGUE (1980) — 코랩판 · 장비 확장

| 파일 | 역할 |
|---|---|
| `rogue_core.py` | 상수 · 등급표 · 장비표 · 클래스 · 던전 생성 |
| `rogue_game.py` | 게임 규칙 (이동 · 근접/원거리 전투 · 장비 · 턴 · 층) |
| 이 노트북 | 화면 그리기 · 키보드 · 실행 |

세 셀을 위에서부터 실행한 뒤, 게임 화면을 한 번 클릭하고 키보드로 조작합니다.

## 기호

```
@ 플레이어      r g o T W G D 몬스터 (깊을수록 강함)
) 근접무기      } 원거리무기     [ 방어구      = 반지
! 물약          $ 골드           ( 화살        ^ 터진 함정
> 계단          * 최종 보물 (10층)
```

## 조작

```
WASD / 방향키  이동 · 이동 방향에 적이 있으면 근접 공격
F              원거리 사격 (가장 가까운, 벽에 막히지 않은 적)
G              발밑 아이템 줍기 / 보물 획득
U              물약 마시기
1~9            가방 칸 사용 · 장착
X              버리기 모드 → 숫자키로 버릴 칸 선택
I              가방 열기/닫기      >  계단 내려가기
.              한 턴 대기          Q  종료      R  다시 시작
```

## 등급

층이 깊어질수록 높은 등급의 장비가 나옵니다.

| 층 | 기준 등급 |
|---|---|
| 1~2 | 녹슨 |
| 3~4 | 평범한+ |
| 5~6 | 강철++ |
| 7~8 | 미스릴★ |
| 9~10 | 전설의★★ |

기준 등급을 중심으로 ±1 등급이 가끔 섞여 나오므로, 2층에서 운 좋게 강철 장비를 줍기도 합니다.

**반지 효과**: 힘(공격력) · 수호(방어력) · 활력(최대 HP) · 재생(주기적 회복) · 행운(치명타율). 두 개까지 동시에 낍니다.

10층에서 `*` 위에 올라가 `G` 를 누르면 클리어.

## 1. 모듈 파일 준비

In [13]:
# 1) rogue_core.py, rogue_game.py 를 /content 에 올린다
#    - 왼쪽 파일 탭에 드래그해도 되고, 아래 업로드 창을 써도 된다
import os

need = [f for f in ("rogue_core.py", "rogue_game.py") if not os.path.exists(f)]

if need:
    print("다음 파일이 없습니다:", need)
    from google.colab import files
    files.upload()
else:
    print("모듈 파일 확인 완료:", os.listdir("."))

다음 파일이 없습니다: ['rogue_game.py']


Saving rogue_game.py to rogue_game.py


## 2. 화면 렌더러

In [14]:
# 2) 모듈 불러오기 + 화면 그리기
import importlib
import html

import rogue_core
import rogue_game

# 모듈 파일을 고친 뒤 이 셀만 다시 실행하면 바로 반영된다
importlib.reload(rogue_core)
importlib.reload(rogue_game)

from rogue_core import (
    MAP_WIDTH, MAP_HEIGHT, MAX_FLOOR,
    TIER_NAMES, floor_base_tier,
    SLOT_MELEE, SLOT_RANGED, SLOT_ARMOR, SLOT_RING1, SLOT_RING2,
)
from rogue_game import Game, MAX_MESSAGES, MAX_INVENTORY

SCREEN_WIDTH = 74


def equipment_lines(game):
    """장착 중인 장비를 두세 줄로 요약."""
    p = game.player

    def short(item, empty="(없음)"):
        if item is None:
            return empty
        if item.item_type == "weapon_ranged":
            return f"{item.name} +{item.value}/{item.attack_range}칸"
        if item.item_type == "ring":
            return f"{item.name} {item.describe().split('(')[-1][:-1]}"
        return f"{item.name} +{item.value}"

    lines = []
    lines.append(
        f"근접  : {short(p.equipment[SLOT_MELEE], '맨손')}"
    )
    lines.append(
        f"원거리: {short(p.equipment[SLOT_RANGED])}   화살 {p.arrows}개"
    )
    lines.append(
        f"방어구: {short(p.equipment[SLOT_ARMOR])}"
    )
    lines.append(
        f"반지1 : {short(p.equipment[SLOT_RING1])}"
    )
    lines.append(
        f"반지2 : {short(p.equipment[SLOT_RING2])}"
    )
    return lines


def inventory_lines(game):
    p = game.player

    lines = []
    header = f"[ 가방 {len(p.inventory)}/{MAX_INVENTORY} ]  숫자키=사용/장착  X=버리기  I=닫기"
    if game.drop_mode:
        header = f"[ 가방 ]  ** 버리기 모드 ** 숫자키로 버릴 칸 선택 (X 취소)"
    lines.append(header)

    if not p.inventory:
        lines.append("  비어 있음")
        return lines

    for i, item in enumerate(p.inventory):
        lines.append(f"  [{i + 1}] {item.symbol} {item.describe()}")

    return lines


def render(game):
    """게임 상태를 하나의 문자열 화면으로 만든다."""
    p = game.player
    lines = []

    lines.append("=" * SCREEN_WIDTH)
    lines.append("ASCII ROGUE".center(SCREEN_WIDTH))
    lines.append(
        f"FLOOR {game.floor} / {MAX_FLOOR}   "
        f"이 층의 기준 등급: {TIER_NAMES[floor_base_tier(game.floor)]}"
        .center(SCREEN_WIDTH)
    )
    lines.append("=" * SCREEN_WIDTH)

    # ----- 던전 -----
    for y in range(MAP_HEIGHT):
        row = []

        for x in range(MAP_WIDTH):

            if not game.is_explored(x, y):
                row.append(" ")
                continue

            if x == p.x and y == p.y:
                row.append("@")
                continue

            monster = game.monster_at(x, y)
            if monster:
                row.append(monster.char)
                continue

            item = game.item_at(x, y)
            if item:
                row.append(item.symbol)
                continue

            trap = game.trap_at(x, y)
            if trap and trap.visible:
                row.append(trap.symbol)
                continue

            if game.stairs and x == game.stairs.x and y == game.stairs.y:
                row.append(game.stairs.symbol)
                continue

            if game.treasure and x == game.treasure.x and y == game.treasure.y:
                row.append(game.treasure.symbol)
                continue

            row.append(game.dungeon[y][x])

        lines.append("".join(row))

    # ----- 상태 -----
    lines.append("-" * SCREEN_WIDTH)

    bar_len = 20
    filled = int(bar_len * p.hp / p.max_hp) if p.max_hp else 0
    bar = "#" * filled + "-" * (bar_len - filled)

    lines.append(f"HP [{bar}] {p.hp}/{p.max_hp}")
    lines.append(
        f"LV {p.level}  XP {p.xp}/{p.next_xp}  "
        f"ATK {p.attack}  DEF {p.defense}  "
        f"치명타 {int(p.crit_chance * 100)}%  GOLD {p.gold}  TURN {game.turn}"
    )

    flags = []
    if p.poisoned:
        flags.append(f"POISON({p.poison_turns})")
    if p.regen_rate:
        flags.append("REGEN")
    if game.is_on_stairs():
        flags.append("계단 위 [>]")
    if game.is_on_treasure():
        flags.append("보물 위 [G]")
    if flags:
        lines.append("상태: " + "   ".join(flags))

    # ----- 장비 -----
    lines.append("-" * SCREEN_WIDTH)
    lines.extend(equipment_lines(game))

    # ----- 메시지 -----
    lines.append("-" * SCREEN_WIDTH)

    messages = game.messages[-MAX_MESSAGES:]
    for message in messages:
        lines.append("> " + message)
    for _ in range(MAX_MESSAGES - len(messages)):
        lines.append("")

    # ----- 가방 -----
    if game.show_inventory:
        lines.append("-" * SCREEN_WIDTH)
        lines.extend(inventory_lines(game))

    # ----- 종료 -----
    if not game.running:
        lines.append("=" * SCREEN_WIDTH)
        if game.won:
            lines.append("GAME CLEAR!".center(SCREEN_WIDTH))
        elif not p.alive:
            lines.append(f"YOU DIED  —  {game.floor}층".center(SCREEN_WIDTH))
        else:
            lines.append("GAME QUIT".center(SCREEN_WIDTH))
        lines.append("R : 다시 시작".center(SCREEN_WIDTH))
        lines.append("=" * SCREEN_WIDTH)

    return "\n".join(lines)


def screen_html(game, error=None):
    """검은 배경의 터미널 화면 HTML."""
    if error is None:
        body = html.escape(render(game))
        color = "#d0ffd0"
    else:
        body = html.escape(error)
        color = "#ff6b6b"

    return f"""
    <pre style="
        margin: 0;
        padding: 16px;
        background: #000000;
        color: {color};
        border: 2px solid #444;
        border-radius: 8px;
        font-family: 'DejaVu Sans Mono', Menlo, Consolas, monospace;
        font-size: 14px;
        line-height: 1.15;
        white-space: pre;
        overflow-x: auto;
    ">{body}</pre>
    """


print("화면 렌더러 준비 완료")

화면 렌더러 준비 완료


## 3. 게임 실행

In [15]:
# 3) 게임 실행 (이 셀을 다시 실행하면 새 게임)
import traceback

import ipywidgets as widgets
from IPython.display import display, Javascript
from google.colab import output

game = Game()

screen = widgets.HTML(
    value=screen_html(game),
    layout=widgets.Layout(width="820px"),
)


def do_key(key):
    """키 하나를 처리하고 화면을 다시 그린다."""
    try:
        game.handle_key(key)
        screen.value = screen_html(game)
    except Exception:
        # 콜백 안에서 난 예외는 코랩이 조용히 삼킨다.
        # 그래서 화면에 직접 찍어준다.
        screen.value = screen_html(game, error=traceback.format_exc())

    return "OK"


output.register_callback("rogue.key", do_key)


def key_button(label, key, width="54px"):
    button = widgets.Button(
        description=label,
        layout=widgets.Layout(width=width, height="32px"),
    )
    button.on_click(lambda _b, k=key: do_key(k))
    return button


pad = widgets.VBox([
    widgets.HBox([
        widgets.Label(layout=widgets.Layout(width="54px")),
        key_button("W", "w"),
    ]),
    widgets.HBox([
        key_button("A", "a"),
        key_button("S", "s"),
        key_button("D", "d"),
    ]),
])

actions = widgets.VBox([
    widgets.HBox([
        key_button("F 사격", "f", "88px"),
        key_button("G 줍기", "g", "88px"),
        key_button("U 물약", "u", "88px"),
    ]),
    widgets.HBox([
        key_button("I 가방", "i", "88px"),
        key_button("X 버리기", "x", "88px"),
        key_button("> 계단", ">", "88px"),
    ]),
    widgets.HBox([
        key_button(". 대기", ".", "88px"),
        key_button("R 재시작", "r", "88px"),
    ]),
])

slots = widgets.HBox(
    [key_button(str(n), str(n), "38px") for n in range(1, 10)]
)

display(widgets.VBox([
    screen,
    widgets.HBox([pad, widgets.Label("  "), actions]),
    widgets.Label("가방 칸 (사용/장착, 버리기 모드에서는 버리기)"),
    slots,
]))


# 키보드 연결 (게임 화면을 한 번 클릭한 뒤 키를 누르세요)
display(Javascript("""
(() => {
    if (window.rogueKeyHandler) {
        document.removeEventListener("keydown", window.rogueKeyHandler, true);
    }

    const validKeys = new Set([
        "w", "a", "s", "d",
        "f", "g", "u", "i", "x", "q", "r",
        ">", ".", " ",
        "1", "2", "3", "4", "5", "6", "7", "8", "9"
    ]);

    window.rogueKeyHandler = function (event) {
        let key = event.key.toLowerCase();

        if (key === "arrowup")    { key = "w"; }
        if (key === "arrowdown")  { key = "s"; }
        if (key === "arrowleft")  { key = "a"; }
        if (key === "arrowright") { key = "d"; }

        if (!validKeys.has(key)) { return; }

        event.preventDefault();
        event.stopPropagation();

        google.colab.kernel.invokeFunction("rogue.key", [key], {});
    };

    document.addEventListener("keydown", window.rogueKeyHandler, true);

    console.log("ROGUE KEYBOARD READY");
})();
"""))

<IPython.core.display.Javascript object>

### 잘 안 될 때

- **키가 안 먹힌다** → 게임 화면을 한 번 클릭해서 출력 영역에 포커스를 준다. 그래도 안 되면 아래 버튼만으로도 전부 플레이할 수 있다.
- **화면이 안 바뀐다** → 멈추는 대신 빨간 글씨로 오류 내용이 화면에 그대로 찍힌다.
- **모듈을 고쳤는데 반영이 안 된다** → 2번 셀부터 다시 실행 (`importlib.reload` 포함).
- **난이도 조절** → `rogue_game.py` 의 `build_floor()` 안 몬스터 수, `rogue_core.py` 의 `create_monster()` 안 `scale` 값.